# Stage 1.4.2.5 — Choosing an OCR Approach and Running OCR
At this point, we already established an important distinction:

A scanned PDF page is essentially an image.
So the normal PDF text extractors we used earlier, such as PyPDFLoader or PyMuPDFLoader, may return little or no meaningful text.
Our goal now is:

            scanned PDF
            ↓
            scanned_page.png
            ↓
            OCR Engine
            ↓
            recognized text
            ↓
            LangChain Document
            ↓
            later → chunking → embedding → vector DB → retrieval

For this learning stage, I recommend that we start with Tesseract OCR.

### 1. Why Tesseract for this stage?
There are several OCR approaches available:

| OCR approach | Type | Best suited for |
| :--- | :--- | :--- |
| Tesseract | Open source | Learning OCR fundamentals, local processing |
| EasyOCR | Open source | Multilingual/general image OCR |
| PaddleOCR | Open source | Stronger document/layout OCR |
| Docling | Open source | Document understanding + structured extraction |
| Azure Document Intelligence | Cloud | Enterprise document processing |
| Google Document AI | Cloud | Enterprise document understanding |
| AWS Textract | Cloud | Enterprise document extraction |
| OpenAI/Gemini multimodal models | Closed/API | Vision + reasoning + extraction |

We shouldn't jump to Docling or cloud OCR yet.
The purpose of Stage 1.4.2.5 is to understand the fundamental mechanism:

How do we turn pixels from a scanned page into text that can enter our RAG pipeline?
Tesseract is excellent for learning that concept.
Tesseract is an open-source OCR engine, and Windows installers are available through the UB Mannheim distribution. The official Tesseract documentation notes that there isn't an official modern Windows installer, while pointing to UB Mannheim for current Windows installers. (GitHub)

### 2. Important: Tesseract has TWO parts
This is something I want you to understand clearly.
When we use:

```python
import pytesseract
we have not installed the OCR engine itself.
There are two components:

Windows
│
├── Tesseract OCR Engine
│       ↓
│   actual OCR program
│
└── Python
        ↓
    pytesseract
        ↓
    Python wrapper that communicates with Tesseract
So:

Component 1 — Tesseract
This is the actual OCR engine.

Component 2 — pytesseract
This is the Python wrapper that allows our notebook to invoke Tesseract.

3. OCR setup on your Windows 10 machine
Because you're using:

Windows 10
VS Code
Notebook
uv
your existing rag-learning project
we'll keep the setup consistent with your existing environment.

Step 3.1 — Install Tesseract itself
Download/install the Windows version from the UB Mannheim Tesseract distribution.
Tesseract Windows installer information — UB Mannheim
During installation, make sure English language data (eng) is installed.
The installer provides language data, and additional language data can also be installed when needed. (GitHub)
A typical installation location is:

C:/Program Files/Tesseract-OCR
and the executable should be:

C:/Program Files/Tesseract-OCR/tesseract.exe
4. Verify Tesseract from PowerShell
After installation, close and reopen VS Code/PowerShell so that any PATH changes are picked up.
Open PowerShell and run:

PowerShell
tesseract --version
You should get something similar to:

tesseract 5.x.x
 leptonica-...
 ...
Then:

PowerShell
tesseract --list-langs
You should see something similar to:

List of available languages in "..."
eng
osd
The eng language model is what we'll use for our scanned page.
The Tesseract documentation also recommends checking the installation with tesseract --version and available languages with tesseract --list-langs. (GitHub)

5. What if PowerShell says tesseract is not recognized?
This means Tesseract is installed but Windows cannot find it through PATH.
You can first test it directly:

PowerShell
& "C:/Program Files/Tesseract-OCR/tesseract.exe" --version
If that works, Tesseract itself is fine.
Then you can add:

C:/Program Files/Tesseract-OCR
to your Windows PATH.
The Tesseract documentation specifically notes that the Tesseract executable directory may need to be added to PATH on Windows. (GitHub)

6. Now install the Python wrapper using UV
Go to your existing RAG project root.
For example:

PowerShell
cd <your-rag-learning-project>
Then:

PowerShell
uv add pytesseract
This is the correct approach for your current project because uv add updates the project's pyproject.toml, lockfile, and environment. (Astral Docs)
We don't need to install Tesseract itself with uv.
Remember:

uv
 │
 └── manages Python dependencies
          ↓
      pytesseract


Windows installer
 │
 └── installs Tesseract OCR engine
That's an important distinction.

7. Verify pytesseract in your notebook
Create a new cell in the notebook you're using for:

Stage 1.4.2.5
Run:


---

In [ ]:
import pytesseract

print(pytesseract.get_tesseract_version())

If everything is configured correctly, you should see the installed Tesseract version.
You can also check:

In [ ]:
print(pytesseract.get_languages())

You should see:


['eng', 'osd', ...]

### 8. Verify our actual scanned_page.png
Now we move from environment setup to the actual RAG learning exercise.
Assuming our previous stage produced:

scanned_page.png
we first load it.

In [ ]:
from PIL import Image

image = Image.open("D:/AI Learning/rag-learning/notebooks/PDF-loaders/extracted_images/page_1_img_1.png")

print(image.size)
print(image.mode)

In [ ]:
display(image)

We want to understand:

What does the OCR engine actually see?
Remember:

PDF viewer:
"I see a page containing text."

OCR engine:
"I see pixels."
OCR must infer characters from those pixels.

### 10. Run our first OCR
Now the interesting part.

In [12]:
import pytesseract

ocr_text = pytesseract.image_to_string(
    image,
    lang="eng"
)

print(ocr_text)

Azure Event Hubs - Scanned Knowledge Page

Azure Event Hubs is a managed real-time event streaming service.
It can ingest large volumes of events from applications, devices,
telemetry systems, and other data sources.

Partitions provide ordered sequences of events and help distribute
processing work across consumers. Consumer groups allow multiple
applications to independently read the same event stream.

This page is intentionally stored as an image inside a PDF.
There is no selectable PDF text layer. A human can read the page,
but a normal text extraction library may return little or no text.



That's our first actual OCR pipeline.
Conceptually:

        scanned_page.png
        │
        ▼
        Image
        │
        ▼
        pytesseract
        │
        ▼
        Tesseract OCR
        │
        ▼
recognized text

### 11. Save the OCR result
For learning purposes, let's preserve the output.

In [ ]:
ocr_output_path = "scanned_page_ocr.txt"

with open(ocr_output_path, "w", encoding="utf-8") as f:
    f.write(ocr_text)

print(f"OCR output saved to: {ocr_output_path}")

Now our notebook directory contains something conceptually like:

scanned_page.png
scanned_page_ocr.txt


This makes the transformation visible:

IMAGE                         TEXT

scanned_page.png      →      scanned_page_ocr.txt

### 12. Now connect OCR to LangChain
This is the RAG-specific part of today's lesson.
OCR by itself isn't RAG.
OCR is an ingestion capability.
Our RAG pipeline becomes:

             Scanned PDF
                  │
                  ▼
          PDF page image
                  │
                  ▼
            OCR Engine
                  │
                  ▼
              OCR text
                  │
                  ▼
       LangChain Document
                  │
                  ▼
              Chunking
                  │
                  ▼
             Embedding
                  │
                  ▼
            Vector Store
                  │
                  ▼
             Retrieval
                  │
                  ▼
                 LLM

This is exactly why OCR matters to RAG.
Without OCR:

Scanned PDF
↓
No extractable text
↓
No meaningful chunks
↓
No useful embeddings
↓
Poor/no retrieval


With OCR:

     Scanned PDF
     ↓
     OCR
     ↓
     Text
     ↓
     Chunks
     ↓
     Embeddings
     ↓
     Vector DB
     ↓
     Retrieval

### 13. Create a LangChain Document
Now we can convert the OCR result into the same object we've already been learning about.

In [ ]:
from langchain_core.documents import Document

ocr_document = Document(
    page_content=ocr_text,
    metadata={
        "source": "scanned_page.png",
        "document_type": "scanned_pdf_page",
        "ocr": True,
        "ocr_engine": "tesseract",
        "language": "eng"
    }
)

Then:

In [ ]:
print(ocr_document)

And:

In [ ]:
print(ocr_document.page_content)

### 14. Why metadata becomes especially important here
Notice what we've added:

```python
"ocr": True

This is useful in a production RAG system.
Suppose later you have:

Document A → native PDF text
Document B → OCR text
Document C → DOCX text
Document D → HTML
Document E → image
You can preserve provenance:

source
file_type
page_number
ocr
ocr_engine
language
For example:

Python
metadata={
    "source": "employee_handbook.pdf",
    "page": 12,
    "ocr": True,
    "ocr_engine": "tesseract",
    "language": "eng"
}
Later, when the answer is generated, you can trace:

LLM answer
   ↓
retrieved chunk
   ↓
OCR-generated chunk
   ↓
page 12
   ↓
employee_handbook.pdf
That's an important production-RAG concept.

15. One very important observation
Don't assume that:

ocr_text
is perfect.
OCR can produce errors such as:

Azure Event Hub
becoming:

Azure Event Huh
or:

Data Explorer
becoming:

Data ExpIorer
This matters enormously for RAG.
Therefore, OCR introduces another quality problem:

Scanned document
       ↓
OCR
       ↓
OCR errors
       ↓
bad text
       ↓
bad chunks
       ↓
bad embeddings
       ↓
bad retrieval
       ↓
bad answer
This is why production RAG systems often have an OCR quality/preprocessing stage.

16. Our first OCR experiment should be deliberately simple
I recommend that Stage 1.4.2.5 should NOT yet become an advanced OCR preprocessing lesson.
First establish the baseline:

Experiment 1

Python
ocr_text = pytesseract.image_to_string(
    image,
    lang="eng"
)
Then inspect:

How much text was recognized?
Are words correct?
Are lines preserved?
Are headings preserved?
Are tables recognized correctly?
Are numbers correct?
Are special characters correct?
Only after seeing the baseline do we introduce:

image preprocessing
       ↓
grayscale
       ↓
noise removal
       ↓
thresholding
       ↓
deskewing
       ↓
resize
       ↓
      OCR
That will make the next lesson much more meaningful.

17. Also understand what Tesseract is NOT doing
Tesseract is primarily answering:

"What characters/words are present in this image?"
It is not necessarily understanding the document semantically.
For example, given:

Employee Details

Name        Department       Experience
-----------------------------------------
Ravi        Engineering      8
Suresh      Data             10
OCR may recognize the words, but understanding:

Name = Ravi
Department = Engineering
Experience = 8
is a document understanding/layout extraction problem.
That's where technologies such as:

PaddleOCR
Docling
Azure Document Intelligence
Google Document AI
AWS Textract
become increasingly relevant.
We'll study those separately rather than mixing them into this first OCR exercise.

18. Your Stage 1.4.2.5 learning target
I suggest we define the stage like this:

Stage 1.4.2.5
│
├── Understand OCR
│
├── Choose Tesseract
│
├── Install Tesseract on Windows
│
├── Install pytesseract using UV
│
├── Verify Tesseract
│
├── Load scanned_page.png
│
├── Run first OCR
│
├── Inspect OCR output
│
├── Save OCR text
│
└── Convert OCR output into
    LangChain Document
And stop there initially.
Then the next natural progression is:

Stage 1.4.2.5
Basic OCR
      ↓
Stage 1.4.2.6
OCR preprocessing
      ↓
Stage 1.4.2.7
OCR quality evaluation
      ↓
Stage 1.4.2.8
OCR + PDF page pipeline
      ↓
Stage 1.4.2.9
OCR → LangChain Documents
We can adjust the numbering to match your existing roadmap, but conceptually this is the right progression.
One thing I want you to do first
Don't start writing the notebook code yet.
First install Tesseract OCR on Windows, then run these two PowerShell commands:

PowerShell
tesseract --version
and:

PowerShell
tesseract --list-langs
Then add the Python dependency from your RAG project's root:

PowerShell
uv add pytesseract
Once those three checks work, your environment is ready and we can execute the OCR against your actual scanned_page.png cell by cell, just as we've been doing throughout your notebook-first RAG journey.
Tesseract Windows installation information
Tesseract official downloads/documentation
UV uv add documentation.